[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/royce-psdi-crystallm-training/blob/master/notebooks/load_and_generate_colab.ipynb)

# Load and Generate Crystal Structures using CrystaLLM

In this notebook, we will use pretrained CrystaLLM models to generate crystal structures in Crystallographic Information File (CIF) format.

Hopefully by the end of this session you will be able to:
1. load a pretrained CrystaLLM model
2. generate crystal structures using different levels of input information
3. locate and examine the generated CIF files
4. visualise the generated crystal structures
5. explore the additional information used by other prompt levels

In [ ]:
# Colab setup for CrystaLLM-pi
# Before running: Runtime → Change runtime type → T4 GPU or better.
# Run this cell once and then restart the session under the Runtime tab above.

%pip install -q uv

%cd /content
![ -d /content/CrystaLLM-pi/.git ] || git clone --depth 1 https://github.com/C-Bone-UCL/CrystaLLM-pi.git

%cd /content/CrystaLLM-pi

!uv pip install --system -r requirements.txt
!uv pip install --system \
    "git+https://github.com/lematerial/material-hasher.git" \
    "git+https://github.com/KellerJordan/Muon"
!uv pip install --system -e .

import os, sys
print("cwd:", os.getcwd())
print("python:", sys.version)

In [ ]:
# Reinstall torch natively

%pip uninstall -y torchvision torchaudio

import torch
import importlib.util

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("torchvision installed:", importlib.util.find_spec("torchvision") is not None)

In [ ]:
# Notebook imports and environment cleanup
%cd /content/CrystaLLM-pi

import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning,ignore::UserWarning"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import pandas as pd


## Part 1. Generating crystal structures with a pretrained base model

We will start with the pretrained CrystaLLM base model.

First, we will generate a crystal structure using the minimum amount of input. Before adding any generation condition, it is useful to see what the original base model produces on its own.
We will then add more information to the input and see how this changes the generation process.

### 1. Load a pretrained model

In [ ]:
# load the pretrained CrystaLLM base model
BASE_MODEL = "c-bone/CrystaLLM-pi_mp_20_base"
print(f"Base model: {BASE_MODEL}")

### 2.  Create Prompts + Generate CIFs

Now that we have selected the pretrained base model, lets have a look at what it generates without any additional constraints.

We will use a Level 1 prompt. At `level_1`, the prompt only contains `data_`so the model is free to generate both the composition and the crystal structure. Therefore, the model has to decide what to generate from the patterns it learned during pretraining.

I guess we could think of this as asking someone to “draw a crystal structure” without giving any further instructions. The answers may vary, but together they show the model's natural starting distribution.


In [ ]:
from pathlib import Path

BASE_OUTPUT = Path("data/workshop_generation/pretrained_base_output.parquet")
BASE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

The generation proceeds in batches.

`num_return_sequences` controls how many candidates are sampled in each batch

`max_return_attempts` sets the maximum number of batches

`target_valid_cifs` tells the pipeline to stop once the requested number of structures has passed its built-in validity checks\

Here, the model samples 20 candidates per batch and it stops once 20 valid CIFs have been collected, or after 20 batches if the tearget has been reached. Since we are using a Level 1 prompt so the generation is unconditional. Feel free to experiment with these settings and see how they affect the generated outputs :）

Also, “valid” only means that the CIF passes the pipeline's built-in checks. It does not guarantee thermodynamic stability.

In [ ]:
!{sys.executable} _load_and_generate.py \
    --hf_model_path {BASE_MODEL} \
    --level level_1 \
    --num_return_sequences 20 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --output_parquet {BASE_OUTPUT}


### 3. Visualise the results

The generated structures and their associated information are stored in a Parquet file so we can load the file as a pandas DataFrame.

Lets look at the first few results.

In [ ]:
import pandas as pd

base_results = pd.read_parquet(BASE_OUTPUT)

print(f"Number of valid CIFs generated: {len(base_results)}")
print(base_results.columns.to_list())

base_results[["Material ID", "Prompt", "Generated CIF"]].head()

#### Take a look at the generated structure

The generated CIFs are currently stored as text in the dataframe. We can read them as `pymatgen` structures and plot a few examples to see what the model has produced.

> **Try it:** Change `random_state` and rerun the cell to have a look at a different set of generated structures.

In [ ]:
import random
import matplotlib.pyplot as plt
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from ase.visualize.plot import plot_atoms

samples = base_results["Generated CIF"].sample(n=3,random_state=5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, cif in zip(axes, samples):
    structure = Structure.from_str(cif, fmt="cif")
    atoms = AseAtomsAdaptor.get_atoms(structure)

    plot_atoms(atoms, ax, radii=0.5)
    ax.set_title(
        f"{structure.composition.reduced_formula}\n"
        f"{structure.density:.2f} g/cm³"
    )
    ax.axis("off")

plt.show()

#### Density Distribution

We can also take a look at the density distribution

In [ ]:
base_densities = []

for cif in base_results["Generated CIF"]:
    structure = Structure.from_str(cif, fmt="cif")
    base_densities.append(structure.density)

plt.hist(base_densities, bins=10, color="#4C78A8", edgecolor="white")
plt.xlabel("Density (g/cm³)")
plt.ylabel("Number of structures")
plt.title("Pretrained base model")
plt.grid(False)
plt.show()

### Try a different prompt

In this example, we used `level_1`, so the model started from a minimal prompt and decided both the composition and crystal structure itself.

CrystaLLM can also start with more information:

- `level_1`: minimal prompt
- `level_2`: composition
- `level_3`: composition and atomic information
- `level_4`: composition, atomic information and space group

Try changing the prompt level and see how giving the model more information affects what it generates.

## Part 2. Compare a toy density model with a pretrained density model

In Part 1, we used an unconditional base model. It was not given a target density, so it was free to generate structures from the distribution it learned during pretraining.

We will now compare **two density-conditioned models**:
1. The toy model produced in the earlier fine-tuning exercise
2. The fully pretrained density-and-stability model

Both models will be asked to generate structures with a target density of **6.0 g/cm³**. and we want to see how a small workshop fine-tune compares with a model trained more extensively for property-conditioned generation.

To keep the comparison consistent:

- both models use a `level_1` prompt;
- both use the same sampling temperature;
- both generate 50 valid CIFs;
- the density of every generated structure is calculated in the same way.

### 1. Define the models and comparison settings

In [1]:
import json
import pandas as pd
from pathlib import Path

TOY_MODEL = "c-bone/MP-20-Density"
PRETRAINED_DENSITY_MODEL = "c-bone/CrystaLLM-pi_density"

MODEL_REGISTRY = Path("data/my_custom_models.json")
OUTPUT_DIR = Path("data/workshop_generation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOY_OUTPUT = OUTPUT_DIR / "toy_density_model.parquet"
PRETRAINED_DENSITY_OUTPUT = OUTPUT_DIR / "pretrained_density_model.parquet"

print(f"Toy model: {TOY_MODEL}")
print(f"Pretrained density model: {PRETRAINED_DENSITY_MODEL}")

Toy model: c-bone/MP-20-Density
Pretrained density model: c-bone/CrystaLLM-pi_density


### 2. Generate structures with toy model/pretrained density model

**Toy model**

First up is our toy model.

This is the model that was fine-tuned earlier using the small toy dataset of approximately 1,000 crystal structures. It was trained to use density as its generation condition, so we give it one value: 6.0

This tells the model that we would like structures with a target density of 6.0 g/cm³.

The model will not necessarily generate structures with exactly this density every time. That is what we are going to investigate later:) for now, let’s generate 50 valid structures and see what it comes up with.

In [ ]:
!python _load_and_generate.py \
    --hf_model_path {TOY_MODEL} \
    --model_registry {MODEL_REGISTRY} \
    --level level_1 \
    --condition_lists "6.0" \
    --num_return_sequences 20 \
    --max_return_attempts 5 \
    --target_valid_cifs 50 \
    --output_parquet {TOY_OUTPUT}

**Pretrained Density Model**

Next, we will generate another 50 valid structures using the pretrained density model.This model was trained on a much larger dataset than our toy model. The model generates crystal structures based on two target scalar properties: 

1. Density (we set it to the valjue of 6.0 g/cm³ in here )
2. Energy above hull (this is a proxy for thermodynamic stability and we set it to 0.0 eV/atom in here)

Although the model needs both values as input, our comparison in this notebook will focus only on the densities of the generated structures.

We will use the same generation settings as before and collect another 50 valid CIFs. Once both sets are ready, we can compare their density distributions and see whether one model stays closer to the target than the other.

In [ ]:
!python _load_and_generate.py \
    --hf_model_path {PRETRAINED_DENSITY_MODEL} \
    --level level_1 \
    --condition_lists "6.0,0.0" \
    --num_return_sequences 20 \
    --max_return_attempts 5 \
    --target_valid_cifs 50 \
    --output_parquet {PRETRAINED_DENSITY_OUTPUT}

### 3. Compare the density distributions

Now we have set of structures from each model. But even though we asked both models for a target density of 6.0 g/cm³, this doesnt mean that every generated structure will have exactly that density. 

So lets calculate the actual density of each generated CIF and see what the two models really gave us.

In [ ]:
from pymatgen.core import Structure

toy_results = pd.read_parquet(TOY_OUTPUT)
pretrained_results = pd.read_parquet(PRETRAINED_DENSITY_OUTPUT)

toy_densities = []
for cif in toy_results["Generated CIF"]:
    structure = Structure.from_str(cif, fmt="cif")
    toy_densities.append(structure.density)

pretrained_densities = []
for cif in pretrained_results["Generated CIF"]:
    structure = Structure.from_str(cif, fmt="cif")
    pretrained_densities.append(structure.density)

print(f"Toy model: {len(toy_results)} structures")
print(f"Pretrained density model: {len(pretrained_results)} structures")

Now lets visualise the results. 

The x-axis shows density and the y-axis shows how many structures fall into each interval. The dashed line shows the target density of 6.0 g/cm³.

Look at the centre and width of each distribution. With 50 structures, we can also see whether either distribution begins to form a clear peak or a bell curve

In [ ]:
import matplotlib.pyplot as plt

plt.hist(
    [toy_densities, pretrained_densities],
    label=["Toy model", "Pretrained density model"],
)

plt.axvline(6.0, color="black", linestyle="--", label="Target density")
plt.xlabel("Density (g/cm³)")
plt.ylabel("Number of structures")
plt.title("Density distribution of generated structures")
plt.legend()
plt.show()

### 4. Look at a few generated structures

The density distribution gives us the overall picture but now lets actually look at a few of the crystals :) We will randomly choose three structures from each model. 

In [ ]:
from pymatgen.io.ase import AseAtomsAdaptor
from ase.visualize.plot import plot_atoms

toy_examples = toy_results["Generated CIF"].sample(3, random_state=5)
pretrained_examples = pretrained_results["Generated CIF"].sample(3, random_state=5)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for ax, cif in zip(axes[0], toy_examples):
    structure = Structure.from_str(cif, fmt="cif")
    
    plot_atoms(AseAtomsAdaptor.get_atoms(structure), ax, radii=0.5)
    ax.set_title(f"Toy model\n{structure.composition.reduced_formula}\n{structure.density:.2f} g/cm³")
    ax.axis("off")

for ax, cif in zip(axes[1], pretrained_examples):
    structure = Structure.from_str(cif, fmt="cif")

    plot_atoms(AseAtomsAdaptor.get_atoms(structure), ax, radii=0.5)
    ax.set_title(f"Pretrained model\n{structure.composition.reduced_formula}\n{structure.density:.2f} g/cm³")
    ax.axis("off")

plt.tight_layout()
plt.show()